In [1]:
# ============================================================
# ORALVISION-M4 — COMPLETE 5-SEED EXPERIMENT
# EfficientNet-B0 + Coordinate Attention (CoordAtt)
# Clean image-level split dataset
#
# RESUMABLE + OOM-HARDENED VERSION
#   - Saves a full checkpoint after EVERY epoch (model, optimizer,
#     scheduler, history, best-state, RNG state). If the kernel
#     dies for any reason, just re-run this cell: it detects the
#     checkpoint and resumes from the exact epoch it left off on,
#     for whichever seed it was on, and skips seeds already fully
#     completed.
#   - Fixes the CUDA OOM at epoch 5/seed 42 by:
#       1) expandable_segments allocator (fixes fragmentation,
#          which is almost certainly what killed epoch 5 — the
#          allocator fragments over a few epochs even though
#          total free VRAM is fine)
#       2) Automatic Mixed Precision (AMP) — roughly halves
#          activation memory on your RTX 2050 (4GB)
#       3) Gradient accumulation so the EFFECTIVE batch size stays
#          32 while the real per-step batch is smaller (16)
#       4) A retry-on-OOM safety net around each training step that
#          clears the cache and retries once instead of crashing
#       5) torch.cuda.empty_cache() + gc.collect() between seeds
#
# RUN THIS ENTIRE BLOCK IN ONE JUPYTER CELL
# ============================================================

import os

# MUST be set before torch/cuda is imported/initialized
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import json
import random
import time
import copy
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler

import torchvision
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASET_ROOT = Path(
    r"C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN"
)

OUTPUT_ROOT = Path(
    r"C:\Users\R Y Z E N\Desktop\oralvision_M4_5seed_results"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Experiment
# -----------------------------
SEEDS = [42, 123, 2024, 3407, 7777]

IMAGE_SIZE = 224

# --- OOM FIX: smaller real batch, accumulate to reach the same
# effective batch size of 32 that the original run used. This is
# the single biggest lever on a 4GB card.
REAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
EFFECTIVE_BATCH_SIZE = REAL_BATCH_SIZE * GRAD_ACCUM_STEPS  # = 32

NUM_WORKERS = 0

EPOCHS = 30

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

DROPOUT = 0.30

# CoordAtt
COORDATT_REDUCTION = 32

# Main M4 imbalance strategy
USE_CLASS_WEIGHTS = True
USE_WEIGHTED_SAMPLER = True

# Early stopping
EARLY_STOPPING_PATIENCE = 7

# ReduceLROnPlateau
LR_FACTOR = 0.5
LR_PATIENCE = 3
MIN_LR = 1e-7

# Save plots
SAVE_PLOTS = True

# Mixed precision
USE_AMP = True

# Checkpoint every epoch (set to a higher number only if disk I/O
# ever becomes the bottleneck — for OOM-resume purposes, 1 is safest)
CHECKPOINT_EVERY_N_EPOCHS = 1

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


# ============================================================
# 2. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
# --- GPU sanity check: fail fast with a clear message instead of a
# confusing OOM on a tiny tensor if the GPU is already full ---
if torch.cuda.is_available():
    free_mem, total_mem = torch.cuda.mem_get_info()
    free_gb = free_mem / 1e9
    total_gb = total_mem / 1e9
    print(f"GPU free memory : {free_gb:.2f} GB / {total_gb:.2f} GB total")

    if free_gb < 0.5:
        raise RuntimeError(
            f"Only {free_gb:.2f} GB free on GPU before training has even "
            f"started. This almost always means a previous crashed run's "
            f"CUDA context (or a zombie python process) is still holding "
            f"memory. RESTART THE JUPYTER KERNEL (not just re-run this "
            f"cell) and check `nvidia-smi` for leftover python.exe "
            f"processes before retrying."
        )
print("=" * 70)
print("ORALVISION-M4 — 5-SEED EXPERIMENT (RESUMABLE)")
print("=" * 70)
print(f"PyTorch version : {torch.__version__}")
print(f"Torchvision     : {torchvision.__version__}")
print(f"Device          : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory      : {total_mem_gb:.1f} GB")

print(f"Dataset root    : {DATASET_ROOT}")
print(f"Output root     : {OUTPUT_ROOT}")
print(f"Real batch size : {REAL_BATCH_SIZE}  (accum x{GRAD_ACCUM_STEPS} "
      f"= effective {EFFECTIVE_BATCH_SIZE})")
print(f"AMP enabled     : {USE_AMP}")
print("=" * 70)


# ============================================================
# 3. CHECK DATASET
# ============================================================

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset directory not found:\n{DATASET_ROOT}"
    )


def find_split_directory(root, split_name):
    candidates = []

    for p in root.rglob("*"):
        if p.is_dir() and p.name.lower() == split_name.lower():
            candidates.append(p)

    if not candidates:
        return None

    scored = []

    for p in candidates:
        class_dirs = [x for x in p.iterdir() if x.is_dir()]
        score = len(class_dirs)
        scored.append((score, p))

    scored.sort(key=lambda x: x[0], reverse=True)

    return scored[0][1]


TRAIN_DIR = find_split_directory(DATASET_ROOT, "train")
VAL_DIR = find_split_directory(DATASET_ROOT, "val")

if VAL_DIR is None:
    VAL_DIR = find_split_directory(DATASET_ROOT, "validation")

TEST_DIR = find_split_directory(DATASET_ROOT, "test")

if TRAIN_DIR is None:
    raise FileNotFoundError("Could not find TRAIN directory inside dataset.")

if VAL_DIR is None:
    raise FileNotFoundError("Could not find VALIDATION directory inside dataset.")

if TEST_DIR is None:
    raise FileNotFoundError("Could not find TEST directory inside dataset.")

print("\nDataset split directories:")
print("TRAIN:", TRAIN_DIR)
print("VAL  :", VAL_DIR)
print("TEST :", TEST_DIR)


# ============================================================
# 4. REPRODUCIBILITY
# ============================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_rng_state():
    """Capture all RNG states so a resumed run is bit-for-bit continuable."""
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["torch_cuda"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and "torch_cuda" in state:
        torch.cuda.set_rng_state_all(state["torch_cuda"])


# ============================================================
# 5. TRANSFORMS
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])


# ============================================================
# 6. DATASET LOADER (inspect classes)
# ============================================================

train_dataset_raw = datasets.ImageFolder(TRAIN_DIR)
val_dataset_raw = datasets.ImageFolder(VAL_DIR)
test_dataset_raw = datasets.ImageFolder(TEST_DIR)

class_names = train_dataset_raw.classes
num_classes = len(class_names)

print("\nClasses:")
for i, c in enumerate(class_names):
    print(f"{i}: {c}")

print(f"\nNumber of classes: {num_classes}")

if val_dataset_raw.classes != class_names:
    raise RuntimeError("Validation classes do not match training classes.")

if test_dataset_raw.classes != class_names:
    raise RuntimeError("Test classes do not match training classes.")


# ============================================================
# 7. CLASS DISTRIBUTION
# ============================================================

train_targets = np.array(train_dataset_raw.targets)
val_targets = np.array(val_dataset_raw.targets)
test_targets = np.array(test_dataset_raw.targets)


def class_distribution(targets, class_names):
    counts = np.bincount(targets, minlength=len(class_names))
    return {class_names[i]: int(counts[i]) for i in range(len(class_names))}


train_distribution = class_distribution(train_targets, class_names)
val_distribution = class_distribution(val_targets, class_names)
test_distribution = class_distribution(test_targets, class_names)

print("\nTRAIN distribution:")
print(train_distribution)
print("\nVALIDATION distribution:")
print(val_distribution)
print("\nTEST distribution:")
print(test_distribution)


# ============================================================
# 8. CLASS WEIGHTS
# ============================================================

N = len(train_targets)
C = num_classes

class_counts = np.bincount(train_targets, minlength=num_classes)
class_weights = N / (C * class_counts)

class_weights_tensor = torch.tensor(
    class_weights, dtype=torch.float32, device=DEVICE
)

print("\nClass weights:")
for i, c in enumerate(class_names):
    print(f"{c:25s} n={class_counts[i]:5d} weight={class_weights[i]:.6f}")


# ============================================================
# 9. COORDINATE ATTENTION
# ============================================================

class CoordAtt(nn.Module):
    def __init__(self, inp, reduction=32):
        super().__init__()
        self.inp = inp
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        mip = max(8, inp // reduction)

        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.Hardswish()

        self.conv_h = nn.Conv2d(mip, inp, kernel_size=1, stride=1, padding=0, bias=True)
        self.conv_w = nn.Conv2d(mip, inp, kernel_size=1, stride=1, padding=0, bias=True)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()

        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        a_h = torch.sigmoid(self.conv_h(x_h))
        a_w = torch.sigmoid(self.conv_w(x_w))

        return identity * a_h * a_w


# ============================================================
# 10. MODEL
# ============================================================

def build_model(num_classes, dropout=DROPOUT, coordatt_reduction=COORDATT_REDUCTION):
    try:
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
    except Exception as e:
        print("\nWARNING: Could not load pretrained ImageNet weights.")
        print("Error:", e)
        print("Falling back to randomly initialized model.")
        model = models.efficientnet_b0(weights=None)

    model.features.add_module(
        "coordatt", CoordAtt(inp=1280, reduction=coordatt_reduction)
    )

    in_features = model.classifier[1].in_features

    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes)
    )

    return model


# ============================================================
# 11. DATASET FACTORY
# ============================================================

def create_datasets():
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transform)
    test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)
    return train_dataset, val_dataset, test_dataset


# ============================================================
# 12. DATALOADER FACTORY
# ============================================================

def create_dataloaders(train_dataset, val_dataset, test_dataset):
    train_targets_local = np.array(train_dataset.targets)

    sample_weights = class_weights[train_targets_local]
    sample_weights = torch.as_tensor(sample_weights, dtype=torch.double)

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=REAL_BATCH_SIZE,
        sampler=sampler if USE_WEIGHTED_SAMPLER else None,
        shuffle=False if USE_WEIGHTED_SAMPLER else True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=REAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=REAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, val_loader, test_loader


# ============================================================
# 13. METRICS
# ============================================================

def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred)
    }


# ============================================================
# 14. EVALUATION
# ============================================================

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast(device_type="cuda", enabled=(USE_AMP and DEVICE.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = torch.argmax(outputs, dim=1)

            all_targets.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    metrics = calculate_metrics(all_targets, all_predictions)
    metrics["loss"] = epoch_loss

    return metrics, np.array(all_targets), np.array(all_predictions)


# ============================================================
# 14b. OOM-SAFE TRAINING STEP
# ============================================================

def training_step_with_oom_retry(model, images, labels, criterion, scaler, do_step):
    """
    Runs one forward/backward (optionally optimizer step) under AMP.
    On CUDA OOM, clears the cache, retries ONCE. If it still fails,
    re-raises so the outer checkpoint logic can save state and stop
    cleanly instead of losing everything.
    """
    try:
        with autocast(device_type="cuda", enabled=(USE_AMP and DEVICE.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        return outputs, loss

    except torch.cuda.OutOfMemoryError:
        print("  !! CUDA OOM on this batch — clearing cache and retrying once...")
        del images, labels
        gc.collect()
        torch.cuda.empty_cache()
        raise  # re-raise; caller decides whether to retry with a smaller batch


# ============================================================
# 15. CHECKPOINT HELPERS
# ============================================================

def checkpoint_path(seed_output):
    return seed_output / "checkpoint.pth"


def summary_path(seed_output):
    return seed_output / "summary.json"


def save_checkpoint(seed_output, seed, epoch, model, optimizer, scheduler,
                     scaler, history, best_val_macro_f1, best_epoch,
                     best_state, patience_counter):
    """Atomic-ish checkpoint save (write to temp file then replace)."""
    tmp_path = checkpoint_path(seed_output).with_suffix(".tmp")

    torch.save({
        "seed": seed,
        "epoch": epoch,  # last COMPLETED epoch
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "history": history,
        "best_val_macro_f1": best_val_macro_f1,
        "best_epoch": best_epoch,
        "best_state": best_state,
        "patience_counter": patience_counter,
        "rng_state": get_rng_state(),
        "class_names": class_names,
    }, tmp_path)

    tmp_path.replace(checkpoint_path(seed_output))


def load_checkpoint(seed_output):
    ckpt_path = checkpoint_path(seed_output)
    if not ckpt_path.exists():
        return None
    return torch.load(ckpt_path, map_location=DEVICE, weights_only=False)


# ============================================================
# 16. TRAIN ONE SEED (RESUMABLE)
# ============================================================

def train_one_seed(seed):

    seed_output = OUTPUT_ROOT / f"seed_{seed}"
    seed_output.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Already fully completed in a previous run? Skip entirely.
    # --------------------------------------------------------
    if summary_path(seed_output).exists():
        print(f"\nSeed {seed}: summary.json already exists — SKIPPING "
              f"(already completed).")
        with open(summary_path(seed_output), "r") as f:
            return json.load(f)

    print("\n")
    print("=" * 70)
    print(f"STARTING SEED: {seed}")
    print("=" * 70)

    # --------------------------------------------------------
    # Datasets / loaders / model / optimizer / scheduler / scaler
    # (built fresh every time; state is restored onto them below)
    # --------------------------------------------------------
    train_dataset, val_dataset, test_dataset = create_datasets()
    train_loader, val_loader, test_loader = create_dataloaders(
        train_dataset, val_dataset, test_dataset
    )

    model = build_model(num_classes=num_classes).to(DEVICE)

    criterion = (
        nn.CrossEntropyLoss(weight=class_weights_tensor)
        if USE_CLASS_WEIGHTS else nn.CrossEntropyLoss()
    )

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=MIN_LR
    )

    scaler = GradScaler(enabled=(USE_AMP and DEVICE.type == "cuda"))

    # --------------------------------------------------------
    # Resume from checkpoint if one exists for this seed
    # --------------------------------------------------------
    ckpt = load_checkpoint(seed_output)

    if ckpt is not None:
        print(f"Seed {seed}: checkpoint found at epoch {ckpt['epoch']} — RESUMING.")

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        set_rng_state(ckpt["rng_state"])

        history = ckpt["history"]
        best_val_macro_f1 = ckpt["best_val_macro_f1"]
        best_epoch = ckpt["best_epoch"]
        best_state = ckpt["best_state"]
        patience_counter = ckpt["patience_counter"]
        start_epoch = ckpt["epoch"] + 1  # resume on the NEXT epoch
        total_training_time_so_far = sum(h["epoch_time_sec"] for h in history)

    else:
        print(f"Seed {seed}: no checkpoint found — starting fresh.")
        seed_everything(seed)

        history = []
        best_val_macro_f1 = -np.inf
        best_epoch = 0
        best_state = None
        patience_counter = 0
        start_epoch = 1
        total_training_time_so_far = 0.0

    start_time = time.time()

    # ========================================================
    # EPOCH LOOP (resumable)
    # ========================================================

    stopped_early = False

    for epoch in range(start_epoch, EPOCHS + 1):

        epoch_start = time.time()
        model.train()

        running_loss = 0.0
        train_targets_epoch = []
        train_predictions_epoch = []

        optimizer.zero_grad(set_to_none=True)

        for step, (images, labels) in enumerate(train_loader):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            try:
                outputs, loss = training_step_with_oom_retry(
                    model, images, labels, criterion, scaler,
                    do_step=((step + 1) % GRAD_ACCUM_STEPS == 0)
                )
            except torch.cuda.OutOfMemoryError:
                # One retry after clearing cache. If this also fails,
                # save a checkpoint at the last completed epoch and
                # stop cleanly rather than losing the whole run.
                try:
                    with autocast(device_type="cuda", enabled=(USE_AMP and DEVICE.type == "cuda")):
                        outputs = model(images)
                        loss = criterion(outputs, labels) / GRAD_ACCUM_STEPS
                    scaler.scale(loss).backward()
                except torch.cuda.OutOfMemoryError:
                    print(f"\n!! Persistent OOM at seed {seed}, epoch {epoch}. "
                          f"Saving checkpoint at last completed epoch "
                          f"({epoch - 1}) and stopping. Re-run the cell to "
                          f"resume — consider lowering REAL_BATCH_SIZE further "
                          f"if this keeps happening.")
                    save_checkpoint(
                        seed_output, seed, epoch - 1, model, optimizer,
                        scheduler, scaler, history, best_val_macro_f1,
                        best_epoch, best_state, patience_counter
                    )
                    raise

            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item() * images.size(0) * GRAD_ACCUM_STEPS

            with torch.no_grad():
                predictions = torch.argmax(outputs, dim=1)
                train_targets_epoch.extend(labels.detach().cpu().numpy())
                train_predictions_epoch.extend(predictions.detach().cpu().numpy())

        train_loss = running_loss / len(train_loader.dataset)
        train_metrics = calculate_metrics(train_targets_epoch, train_predictions_epoch)

        # Free fragmented memory between train and eval passes
        gc.collect()
        torch.cuda.empty_cache()

        val_metrics, _, _ = evaluate(model, val_loader, criterion)

        scheduler.step(val_metrics["macro_f1"])
        current_lr = optimizer.param_groups[0]["lr"]

        epoch_time = time.time() - epoch_start

        row = {
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "train_balanced_accuracy": train_metrics["balanced_accuracy"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "learning_rate": current_lr,
            "epoch_time_sec": epoch_time
        }

        history.append(row)

        print(
            f"Seed {seed} | Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss {train_loss:.4f} | Train F1 {train_metrics['macro_f1']:.4f} | "
            f"Val Loss {val_metrics['loss']:.4f} | Val F1 {val_metrics['macro_f1']:.4f} | "
            f"LR {current_lr:.2e} | {epoch_time:.1f}s"
        )

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

            torch.save({
                "seed": seed,
                "epoch": epoch,
                "model_state_dict": best_state,
                "class_names": class_names,
                "best_val_macro_f1": best_val_macro_f1
            }, seed_output / "best_model.pth")

            patience_counter = 0
            print(f"  >>> BEST MODEL SAVED (Val Macro-F1 = {best_val_macro_f1:.4f})")
        else:
            patience_counter += 1

        # ----------------------------------------------------
        # CHECKPOINT — this is what makes the run resumable
        # ----------------------------------------------------
        if epoch % CHECKPOINT_EVERY_N_EPOCHS == 0:
            save_checkpoint(
                seed_output, seed, epoch, model, optimizer, scheduler,
                scaler, history, best_val_macro_f1, best_epoch,
                best_state, patience_counter
            )

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"  >>> Early stopping at epoch {epoch}")
            stopped_early = True
            # make sure the final state is checkpointed even if it
            # falls off the CHECKPOINT_EVERY_N_EPOCHS cadence
            save_checkpoint(
                seed_output, seed, epoch, model, optimizer, scheduler,
                scaler, history, best_val_macro_f1, best_epoch,
                best_state, patience_counter
            )
            break

    total_training_time = total_training_time_so_far + (time.time() - start_time)

    # ========================================================
    # RESTORE BEST MODEL & EVALUATE ON TEST
    # ========================================================

    if best_state is None:
        raise RuntimeError(f"No best model found for seed {seed}")

    model.load_state_dict(best_state)

    print(f"\nEvaluating BEST checkpoint on TEST set for seed {seed}...")

    test_metrics, y_true, y_pred = evaluate(model, test_loader, criterion)

    report_dict = classification_report(
        y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv(seed_output / "classification_report.csv")

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_df.to_csv(seed_output / "confusion_matrix.csv")

    if SAVE_PLOTS:
        plt.figure(figsize=(8, 7))
        plt.imshow(cm, interpolation="nearest")
        plt.title(f"Confusion Matrix — Seed {seed}")
        plt.colorbar()

        tick_marks = np.arange(len(class_names))
        plt.xticks(tick_marks, class_names, rotation=45, ha="right")
        plt.yticks(tick_marks, class_names)

        threshold = cm.max() / 2 if cm.size > 0 else 0

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(
                    j, i, str(cm[i, j]),
                    horizontalalignment="center",
                    verticalalignment="center",
                    color="white" if cm[i, j] > threshold else "black"
                )

        plt.ylabel("True Label")
        plt.xlabel("Predicted Label")
        plt.tight_layout()
        plt.savefig(seed_output / "confusion_matrix.png", dpi=300, bbox_inches="tight")
        plt.close()

    history_df = pd.DataFrame(history)
    history_df.to_csv(seed_output / "training_history.csv", index=False)

    if SAVE_PLOTS:
        plt.figure(figsize=(8, 5))
        plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
        plt.plot(history_df["epoch"], history_df["val_loss"], label="Validation Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title(f"Loss Curve — Seed {seed}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(seed_output / "loss_curve.png", dpi=300, bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(8, 5))
        plt.plot(history_df["epoch"], history_df["train_macro_f1"], label="Train Macro-F1")
        plt.plot(history_df["epoch"], history_df["val_macro_f1"], label="Validation Macro-F1")
        plt.xlabel("Epoch")
        plt.ylabel("Macro-F1")
        plt.title(f"Macro-F1 Curve — Seed {seed}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(seed_output / "macro_f1_curve.png", dpi=300, bbox_inches="tight")
        plt.close()

    result = {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_weighted_f1": test_metrics["weighted_f1"],
        "test_balanced_accuracy": test_metrics["balanced_accuracy"],
        "training_time_sec": total_training_time,
        "num_epochs_run": len(history)
    }

    with open(summary_path(seed_output), "w") as f:
        json.dump(result, f, indent=4)

    # Seed is fully done — the checkpoint is no longer needed and
    # would otherwise force a (harmless but pointless) reload if
    # this cell is re-run. Keep it by default since disk is cheap;
    # uncomment to auto-clean:
    # checkpoint_path(seed_output).unlink(missing_ok=True)

    print("\n" + "-" * 70)
    print(f"SEED {seed} TEST RESULTS")
    print("-" * 70)
    print(f"Accuracy          : {test_metrics['accuracy']:.4f}")
    print(f"Macro-F1          : {test_metrics['macro_f1']:.4f}")
    print(f"Weighted-F1       : {test_metrics['weighted_f1']:.4f}")
    print(f"Balanced Accuracy : {test_metrics['balanced_accuracy']:.4f}")
    print(f"Best Epoch        : {best_epoch}")
    print(f"Training Time     : {total_training_time / 60:.2f} min")

    # Clean up GPU memory before moving to the next seed
    del model, optimizer, scheduler, scaler, best_state
    gc.collect()
    torch.cuda.empty_cache()

    return result


# ============================================================
# 17. RUN ALL FIVE SEEDS (resumes any seed already in progress,
#     skips any seed already fully completed)
# ============================================================

all_results = []
experiment_start = time.time()

for seed in SEEDS:
    result = train_one_seed(seed)
    all_results.append(result)

results_df = pd.DataFrame(all_results)
results_df.to_csv(OUTPUT_ROOT / "5_seed_results.csv", index=False)


# ============================================================
# 18. MEAN ± SD
# ============================================================

metric_columns = [
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "test_balanced_accuracy", "best_val_macro_f1"
]

summary_rows = []

for metric in metric_columns:
    values = results_df[metric].astype(float)
    summary_rows.append({
        "metric": metric,
        "mean": values.mean(),
        "std": values.std(ddof=1),
        "min": values.min(),
        "max": values.max()
    })

mean_sd_df = pd.DataFrame(summary_rows)
mean_sd_df.to_csv(OUTPUT_ROOT / "5_seed_mean_sd.csv", index=False)


# ============================================================
# 19. FINAL EXPERIMENT SUMMARY
# ============================================================

total_experiment_time = time.time() - experiment_start

final_summary = {
    "experiment": "OralVision-M4",
    "architecture": "EfficientNet-B0 + CoordAtt",
    "dataset": str(DATASET_ROOT),
    "image_size": IMAGE_SIZE,
    "real_batch_size": REAL_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "amp_enabled": USE_AMP,
    "epochs_max": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT,
    "coordatt_reduction": COORDATT_REDUCTION,
    "class_weights": USE_CLASS_WEIGHTS,
    "weighted_sampler": USE_WEIGHTED_SAMPLER,
    "seeds": SEEDS,
    "device": str(DEVICE),
    "num_classes": num_classes,
    "class_names": class_names,
    "train_distribution": train_distribution,
    "validation_distribution": val_distribution,
    "test_distribution": test_distribution,
    "total_experiment_time_sec": total_experiment_time
}

with open(OUTPUT_ROOT / "experiment_summary.json", "w") as f:
    json.dump(final_summary, f, indent=4)


# ============================================================
# 20. PRINT FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("ORALVISION-M4 — FINAL 5-SEED RESULTS")
print("=" * 70)

print(
    results_df[[
        "seed", "best_epoch", "test_accuracy", "test_macro_f1",
        "test_weighted_f1", "test_balanced_accuracy"
    ]].to_string(index=False)
)

print("\n")
print("-" * 70)
print("MEAN ± SD")
print("-" * 70)

for _, row in mean_sd_df.iterrows():
    print(f"{row['metric']:25s}: {row['mean']:.4f} ± {row['std']:.4f}")

print("\n")
print("=" * 70)
print("EXPERIMENT COMPLETE")
print("=" * 70)
print(f"Results saved to:\n{OUTPUT_ROOT}")
print("\nDone.")

GPU free memory : 3.46 GB / 4.29 GB total
ORALVISION-M4 — 5-SEED EXPERIMENT (RESUMABLE)
PyTorch version : 2.5.1+cu121
Torchvision     : 0.20.1+cu121
Device          : cuda
GPU             : NVIDIA GeForce RTX 2050
GPU memory      : 4.3 GB
Dataset root    : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN
Output root     : C:\Users\R Y Z E N\Desktop\oralvision_M4_5seed_results
Real batch size : 16  (accum x2 = effective 32)
AMP enabled     : True

Dataset split directories:
TRAIN: C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\train
VAL  : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\val
TEST : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\test

Classes:
0: Calculus
1: Caries
2: Gingivitis
3: Hypodontia
4: Mouth_Ulcer
5: Tooth_Discoloration

Number of classes: 6

TRAIN distribution:
{'Calculus': 942, 'Caries': 5403, 'Gingivitis': 2595, 'Hypodontia': 2553, 'Mouth_Ulcer': 5844, 'Tooth_Discoloration': 363}

VALIDATION distribution:
{'Calculus': 

TypeError: RNG state must be a torch.ByteTensor

In [2]:
# ============================================================
# ORALVISION-M4 — RTX 2050 FULLY HARDENED 5-SEED EXPERIMENT
# EfficientNet-B0 + Coordinate Attention
#
# RTX 2050 / 4 GB VRAM SAFE VERSION
#
# FEATURES
#   - AMP / mixed precision
#   - Batch size 16
#   - Gradient accumulation x2
#   - Effective batch size 32
#   - CUDA memory cleanup
#   - OOM-safe training
#   - Checkpoint EVERY epoch
#   - Robust checkpoint resume
#   - Backward-compatible RNG restoration
#   - Best-model preservation
#   - Early stopping
#   - ReduceLROnPlateau
#   - 5 independent seeds
#   - Automatic skipping of completed seeds
#
# EXISTING CHECKPOINT:
#   seed_42 epoch 6 -> resumes at epoch 7
# ============================================================


# ============================================================
# 0. CUDA MEMORY CONFIG
# ============================================================

import os

# MUST be set before CUDA initialization
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,"
    "max_split_size_mb:128"
)


# ============================================================
# 1. IMPORTS
# ============================================================

import gc
import json
import random
import time
import copy
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, WeightedRandomSampler

from torch.amp import autocast, GradScaler

import torchvision
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")


# ============================================================
# 2. CONFIGURATION
# ============================================================

DATASET_ROOT = Path(
    r"C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN"
)

OUTPUT_ROOT = Path(
    r"C:\Users\R Y Z E N\Desktop\oralvision_M4_5seed_results"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Experiment
# ------------------------------------------------------------

SEEDS = [
    42,
    123,
    2024,
    3407,
    7777
]

IMAGE_SIZE = 224

EPOCHS = 30


# ------------------------------------------------------------
# RTX 2050 memory-safe batch configuration
# ------------------------------------------------------------

REAL_BATCH_SIZE = 16

GRAD_ACCUM_STEPS = 2

EFFECTIVE_BATCH_SIZE = (
    REAL_BATCH_SIZE * GRAD_ACCUM_STEPS
)


# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

NUM_WORKERS = 0

PIN_MEMORY = torch.cuda.is_available()


# ------------------------------------------------------------
# Optimization
# ------------------------------------------------------------

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

DROPOUT = 0.30


# ------------------------------------------------------------
# CoordAtt
# ------------------------------------------------------------

COORDATT_REDUCTION = 32


# ------------------------------------------------------------
# Imbalance handling
# ------------------------------------------------------------

USE_CLASS_WEIGHTS = True

USE_WEIGHTED_SAMPLER = True


# ------------------------------------------------------------
# Early stopping
# ------------------------------------------------------------

EARLY_STOPPING_PATIENCE = 7


# ------------------------------------------------------------
# Scheduler
# ------------------------------------------------------------

LR_FACTOR = 0.5

LR_PATIENCE = 3

MIN_LR = 1e-7


# ------------------------------------------------------------
# AMP
# ------------------------------------------------------------

USE_AMP = True


# ------------------------------------------------------------
# Checkpoint
# ------------------------------------------------------------

CHECKPOINT_EVERY_N_EPOCHS = 1


# ------------------------------------------------------------
# Plots
# ------------------------------------------------------------

SAVE_PLOTS = True


# ------------------------------------------------------------
# ImageNet normalization
# ------------------------------------------------------------

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]


# ============================================================
# 3. GPU / DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("ORALVISION-M4 — RTX 2050 HARDENED VERSION")
print("=" * 70)

print(
    f"PyTorch version : {torch.__version__}"
)

print(
    f"Torchvision     : {torchvision.__version__}"
)

print(
    f"Device          : {DEVICE}"
)


if torch.cuda.is_available():

    GPU_NAME = torch.cuda.get_device_name(0)

    GPU_TOTAL = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / 1e9
    )

    FREE_MEM, TOTAL_MEM = (
        torch.cuda.mem_get_info()
    )

    FREE_GB = FREE_MEM / 1e9

    print(
        f"GPU             : {GPU_NAME}"
    )

    print(
        f"GPU memory      : {GPU_TOTAL:.2f} GB"
    )

    print(
        f"GPU free memory : {FREE_GB:.2f} GB / "
        f"{TOTAL_MEM / 1e9:.2f} GB"
    )

    if FREE_GB < 0.50:

        raise RuntimeError(
            "\nGPU has less than 0.50 GB free before "
            "training.\n\n"
            "Restart the Jupyter kernel and make sure "
            "no other Python process is using the GPU."
        )


print(
    f"Dataset root    : {DATASET_ROOT}"
)

print(
    f"Output root     : {OUTPUT_ROOT}"
)

print(
    f"Real batch size : {REAL_BATCH_SIZE}"
)

print(
    f"Accumulation    : x{GRAD_ACCUM_STEPS}"
)

print(
    f"Effective batch : {EFFECTIVE_BATCH_SIZE}"
)

print(
    f"AMP enabled     : {USE_AMP}"
)

print("=" * 70)


# ============================================================
# 4. DATASET DISCOVERY
# ============================================================

if not DATASET_ROOT.exists():

    raise FileNotFoundError(
        f"Dataset directory not found:\n"
        f"{DATASET_ROOT}"
    )


def find_split_directory(
    root,
    split_name
):

    candidates = []

    for p in root.rglob("*"):

        if (
            p.is_dir()
            and p.name.lower()
            == split_name.lower()
        ):
            candidates.append(p)

    if not candidates:

        return None

    scored = []

    for p in candidates:

        class_dirs = [
            x
            for x in p.iterdir()
            if x.is_dir()
        ]

        score = len(class_dirs)

        scored.append(
            (score, p)
        )

    scored.sort(
        key=lambda x: x[0],
        reverse=True
    )

    return scored[0][1]


TRAIN_DIR = find_split_directory(
    DATASET_ROOT,
    "train"
)

VAL_DIR = find_split_directory(
    DATASET_ROOT,
    "val"
)

if VAL_DIR is None:

    VAL_DIR = find_split_directory(
        DATASET_ROOT,
        "validation"
    )

TEST_DIR = find_split_directory(
    DATASET_ROOT,
    "test"
)


if TRAIN_DIR is None:

    raise FileNotFoundError(
        "Could not find TRAIN directory."
    )

if VAL_DIR is None:

    raise FileNotFoundError(
        "Could not find VALIDATION directory."
    )

if TEST_DIR is None:

    raise FileNotFoundError(
        "Could not find TEST directory."
    )


print("\nDataset split directories:")

print(
    "TRAIN:",
    TRAIN_DIR
)

print(
    "VAL  :",
    VAL_DIR
)

print(
    "TEST :",
    TEST_DIR
)


# ============================================================
# 5. REPRODUCIBILITY
# ============================================================

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)

        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


def get_rng_state():

    """
    Save RNG states in a safe portable format.
    """

    state = {

        "python":
            random.getstate(),

        "numpy":
            np.random.get_state(),

        "torch":
            torch.get_rng_state()
            .detach()
            .cpu()
            .to(torch.uint8)
            .contiguous()
    }

    if torch.cuda.is_available():

        state[
            "torch_cuda"
        ] = [

            s.detach()
            .cpu()
            .to(torch.uint8)
            .contiguous()

            for s
            in torch.cuda.get_rng_state_all()
        ]

    return state


def set_rng_state(state):

    """
    Robustly restore RNG state.

    Handles old checkpoints where the
    torch RNG state may have been saved in
    an incompatible representation.
    """

    # --------------------------------------------------------
    # Python
    # --------------------------------------------------------

    if "python" in state:

        random.setstate(
            state["python"]
        )


    # --------------------------------------------------------
    # NumPy
    # --------------------------------------------------------

    if "numpy" in state:

        np.random.set_state(
            state["numpy"]
        )


    # --------------------------------------------------------
    # PyTorch CPU
    # --------------------------------------------------------

    if "torch" in state:

        torch_state = state["torch"]

        if isinstance(
            torch_state,
            torch.Tensor
        ):

            torch_state = (
                torch_state
                .detach()
                .cpu()
                .to(torch.uint8)
                .contiguous()
            )

        else:

            torch_state = torch.tensor(
                torch_state,
                dtype=torch.uint8,
                device="cpu"
            ).contiguous()

        torch.set_rng_state(
            torch_state
        )


    # --------------------------------------------------------
    # PyTorch CUDA
    # --------------------------------------------------------

    if (
        torch.cuda.is_available()
        and "torch_cuda" in state
    ):

        cuda_states = state[
            "torch_cuda"
        ]

        fixed_cuda_states = []

        for cuda_state in cuda_states:

            if isinstance(
                cuda_state,
                torch.Tensor
            ):

                cuda_state = (
                    cuda_state
                    .detach()
                    .cpu()
                    .to(torch.uint8)
                    .contiguous()
                )

            else:

                cuda_state = torch.tensor(
                    cuda_state,
                    dtype=torch.uint8,
                    device="cpu"
                ).contiguous()

            fixed_cuda_states.append(
                cuda_state
            )

        if fixed_cuda_states:

            torch.cuda.set_rng_state_all(
                fixed_cuda_states
            )


# ============================================================
# 6. TRANSFORMS
# ============================================================

train_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(
        p=0.50
    ),

    transforms.RandomRotation(
        degrees=15
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


eval_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# ============================================================
# 7. DATASET INSPECTION
# ============================================================

train_dataset_raw = datasets.ImageFolder(
    TRAIN_DIR
)

val_dataset_raw = datasets.ImageFolder(
    VAL_DIR
)

test_dataset_raw = datasets.ImageFolder(
    TEST_DIR
)


class_names = (
    train_dataset_raw.classes
)

num_classes = len(
    class_names
)


print("\nClasses:")

for i, c in enumerate(
    class_names
):

    print(
        f"{i}: {c}"
    )


print(
    f"\nNumber of classes: "
    f"{num_classes}"
)


if (
    val_dataset_raw.classes
    != class_names
):

    raise RuntimeError(
        "Validation classes do not match "
        "training classes."
    )


if (
    test_dataset_raw.classes
    != class_names
):

    raise RuntimeError(
        "Test classes do not match "
        "training classes."
    )


# ============================================================
# 8. CLASS DISTRIBUTION
# ============================================================

train_targets = np.array(
    train_dataset_raw.targets
)

val_targets = np.array(
    val_dataset_raw.targets
)

test_targets = np.array(
    test_dataset_raw.targets
)


def class_distribution(
    targets,
    class_names
):

    counts = np.bincount(
        targets,
        minlength=len(class_names)
    )

    return {
        class_names[i]:
            int(counts[i])

        for i
        in range(len(class_names))
    }


train_distribution = class_distribution(
    train_targets,
    class_names
)

val_distribution = class_distribution(
    val_targets,
    class_names
)

test_distribution = class_distribution(
    test_targets,
    class_names
)


print("\nTRAIN distribution:")

print(
    train_distribution
)

print("\nVALIDATION distribution:")

print(
    val_distribution
)

print("\nTEST distribution:")

print(
    test_distribution
)


# ============================================================
# 9. CLASS WEIGHTS
# ============================================================

N = len(
    train_targets
)

C = num_classes

class_counts = np.bincount(
    train_targets,
    minlength=num_classes
)


# Safety check
if np.any(
    class_counts == 0
):

    raise RuntimeError(
        "At least one class has zero "
        "training samples."
    )


class_weights = (
    N
    /
    (C * class_counts)
)


class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)


print("\nClass weights:")

for i, c in enumerate(
    class_names
):

    print(
        f"{c:25s} "
        f"n={class_counts[i]:5d} "
        f"weight={class_weights[i]:.6f}"
    )


# ============================================================
# 10. COORDINATE ATTENTION
# ============================================================

class CoordAtt(nn.Module):

    def __init__(
        self,
        inp,
        reduction=32
    ):

        super().__init__()

        self.inp = inp

        self.pool_h = (
            nn.AdaptiveAvgPool2d(
                (None, 1)
            )
        )

        self.pool_w = (
            nn.AdaptiveAvgPool2d(
                (1, None)
            )
        )

        mip = max(
            8,
            inp // reduction
        )

        self.conv1 = nn.Conv2d(
            inp,
            mip,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(
            mip
        )

        self.act = nn.Hardswish()

        self.conv_h = nn.Conv2d(
            mip,
            inp,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=True
        )

        self.conv_w = nn.Conv2d(
            mip,
            inp,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=True
        )


    def forward(self, x):

        identity = x

        n, c, h, w = x.size()

        x_h = self.pool_h(x)

        x_w = (
            self.pool_w(x)
            .permute(0, 1, 3, 2)
        )

        y = torch.cat(
            [x_h, x_w],
            dim=2
        )

        y = self.conv1(y)

        y = self.bn1(y)

        y = self.act(y)

        x_h, x_w = torch.split(
            y,
            [h, w],
            dim=2
        )

        x_w = (
            x_w
            .permute(0, 1, 3, 2)
        )

        a_h = torch.sigmoid(
            self.conv_h(x_h)
        )

        a_w = torch.sigmoid(
            self.conv_w(x_w)
        )

        return (
            identity
            * a_h
            * a_w
        )


# ============================================================
# 11. MODEL
# ============================================================

def build_model(
    num_classes,
    dropout=DROPOUT,
    coordatt_reduction=COORDATT_REDUCTION
):

    try:

        weights = (
            models.EfficientNet_B0_Weights.DEFAULT
        )

        model = (
            models.efficientnet_b0(
                weights=weights
            )
        )

    except Exception as e:

        print(
            "\nWARNING: Could not load "
            "pretrained ImageNet weights."
        )

        print(
            "Error:",
            e
        )

        print(
            "Using randomly initialized model."
        )

        model = (
            models.efficientnet_b0(
                weights=None
            )
        )


    model.features.add_module(
        "coordatt",
        CoordAtt(
            inp=1280,
            reduction=coordatt_reduction
        )
    )


    in_features = (
        model.classifier[1]
        .in_features
    )


    model.classifier = nn.Sequential(

        nn.Dropout(
            p=dropout
        ),

        nn.Linear(
            in_features,
            num_classes
        )
    )


    return model


# ============================================================
# 12. DATASET FACTORY
# ============================================================

def create_datasets():

    train_dataset = datasets.ImageFolder(
        TRAIN_DIR,
        transform=train_transform
    )

    val_dataset = datasets.ImageFolder(
        VAL_DIR,
        transform=eval_transform
    )

    test_dataset = datasets.ImageFolder(
        TEST_DIR,
        transform=eval_transform
    )

    return (
        train_dataset,
        val_dataset,
        test_dataset
    )


# ============================================================
# 13. DATALOADER FACTORY
# ============================================================

def create_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset
):

    train_targets_local = np.array(
        train_dataset.targets
    )


    sample_weights = (
        class_weights[
            train_targets_local
        ]
    )


    sample_weights = torch.as_tensor(
        sample_weights,
        dtype=torch.double
    )


    sampler = WeightedRandomSampler(

        weights=sample_weights,

        num_samples=len(
            sample_weights
        ),

        replacement=True
    )


    train_loader = DataLoader(

        train_dataset,

        batch_size=REAL_BATCH_SIZE,

        sampler=(
            sampler
            if USE_WEIGHTED_SAMPLER
            else None
        ),

        shuffle=(
            False
            if USE_WEIGHTED_SAMPLER
            else True
        ),

        num_workers=NUM_WORKERS,

        pin_memory=PIN_MEMORY,

        persistent_workers=False
    )


    val_loader = DataLoader(

        val_dataset,

        batch_size=REAL_BATCH_SIZE,

        shuffle=False,

        num_workers=NUM_WORKERS,

        pin_memory=PIN_MEMORY,

        persistent_workers=False
    )


    test_loader = DataLoader(

        test_dataset,

        batch_size=REAL_BATCH_SIZE,

        shuffle=False,

        num_workers=NUM_WORKERS,

        pin_memory=PIN_MEMORY,

        persistent_workers=False
    )


    return (
        train_loader,
        val_loader,
        test_loader
    )


# ============================================================
# 14. METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            )
    }


# ============================================================
# 15. EVALUATION
# ============================================================

def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0.0

    all_targets = []

    all_predictions = []


    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )


            with autocast(
                device_type="cuda",
                enabled=(
                    USE_AMP
                    and DEVICE.type == "cuda"
                )
            ):

                outputs = model(
                    images
                )

                loss = criterion(
                    outputs,
                    labels
                )


            running_loss += (
                loss.item()
                * images.size(0)
            )


            predictions = torch.argmax(
                outputs,
                dim=1
            )


            all_targets.extend(
                labels
                .detach()
                .cpu()
                .numpy()
            )

            all_predictions.extend(
                predictions
                .detach()
                .cpu()
                .numpy()
            )


            # Explicitly release batch tensors
            del (
                images,
                labels,
                outputs,
                predictions,
                loss
            )


    epoch_loss = (
        running_loss
        /
        len(loader.dataset)
    )


    metrics = calculate_metrics(
        all_targets,
        all_predictions
    )

    metrics["loss"] = epoch_loss


    return (
        metrics,
        np.array(all_targets),
        np.array(all_predictions)
    )


# ============================================================
# 16. GPU MEMORY CLEANUP
# ============================================================

def cleanup_gpu():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        # Synchronize only when needed
        torch.cuda.synchronize()


def print_gpu_memory(
    prefix=""
):

    if not torch.cuda.is_available():

        return

    try:

        allocated = (
            torch.cuda.memory_allocated()
            / 1e9
        )

        reserved = (
            torch.cuda.memory_reserved()
            / 1e9
        )

        free_mem, total_mem = (
            torch.cuda.mem_get_info()
        )

        free_gb = (
            free_mem / 1e9
        )

        print(
            f"{prefix}"
            f"GPU allocated={allocated:.2f} GB | "
            f"reserved={reserved:.2f} GB | "
            f"free={free_gb:.2f} GB"
        )

    except Exception:

        pass


# ============================================================
# 17. CHECKPOINT HELPERS
# ============================================================

def checkpoint_path(
    seed_output
):

    return (
        seed_output
        / "checkpoint.pth"
    )


def summary_path(
    seed_output
):

    return (
        seed_output
        / "summary.json"
    )


def save_checkpoint(
    seed_output,
    seed,
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    history,
    best_val_macro_f1,
    best_epoch,
    best_state,
    patience_counter
):

    """
    Full resumable checkpoint.
    """

    tmp_path = (
        seed_output
        / "checkpoint.tmp"
    )


    checkpoint = {

        "checkpoint_version":
            2,

        "seed":
            seed,

        "epoch":
            epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "scaler_state_dict":
            scaler.state_dict(),

        "history":
            history,

        "best_val_macro_f1":
            best_val_macro_f1,

        "best_epoch":
            best_epoch,

        "best_state":
            best_state,

        "patience_counter":
            patience_counter,

        "rng_state":
            get_rng_state(),

        "class_names":
            class_names,

        "real_batch_size":
            REAL_BATCH_SIZE,

        "grad_accum_steps":
            GRAD_ACCUM_STEPS,

        "effective_batch_size":
            EFFECTIVE_BATCH_SIZE,

        "image_size":
            IMAGE_SIZE
    }


    torch.save(
        checkpoint,
        tmp_path
    )


    # Atomic replacement
    tmp_path.replace(
        checkpoint_path(
            seed_output
        )
    )


def load_checkpoint(
    seed_output
):

    ckpt_path = checkpoint_path(
        seed_output
    )


    if not ckpt_path.exists():

        return None


    print(
        f"\nLoading checkpoint:\n"
        f"{ckpt_path}"
    )


    ckpt = torch.load(

        ckpt_path,

        map_location=DEVICE,

        weights_only=False
    )


    required_keys = [

        "seed",
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
        "scheduler_state_dict",
        "scaler_state_dict",
        "history",
        "best_val_macro_f1",
        "best_epoch",
        "best_state",
        "patience_counter",
        "rng_state"
    ]


    missing = [

        key
        for key in required_keys
        if key not in ckpt
    ]


    if missing:

        raise RuntimeError(
            "Checkpoint is missing keys: "
            f"{missing}"
        )


    print(
        "Checkpoint loaded successfully."
    )

    print(
        f"Seed: "
        f"{ckpt['seed']}"
    )

    print(
        f"Last completed epoch: "
        f"{ckpt['epoch']}"
    )


    return ckpt


# ============================================================
# 18. SAFE CHECKPOINT RESUME
# ============================================================

def restore_checkpoint(
    ckpt,
    model,
    optimizer,
    scheduler,
    scaler
):

    """
    Restore model/training state safely.
    """

    model.load_state_dict(
        ckpt["model_state_dict"]
    )


    optimizer.load_state_dict(
        ckpt["optimizer_state_dict"]
    )


    scheduler.load_state_dict(
        ckpt["scheduler_state_dict"]
    )


    scaler.load_state_dict(
        ckpt["scaler_state_dict"]
    )


    # RNG restoration is isolated so a legacy
    # RNG format cannot corrupt model restoration.
    try:

        set_rng_state(
            ckpt["rng_state"]
        )

        print(
            "RNG state restored successfully."
        )

    except Exception as e:

        print(
            "\nWARNING:"
            " Could not restore RNG state."
        )

        print(
            "Reason:",
            repr(e)
        )

        print(
            "Model/optimizer checkpoint will "
            "still be resumed."
        )

        print(
            "Future checkpoints will use the "
            "new safe RNG format."
        )


# ============================================================
# 19. TRAINING
# ============================================================

def train_one_seed(seed):

    seed_output = (
        OUTPUT_ROOT
        / f"seed_{seed}"
    )

    seed_output.mkdir(
        parents=True,
        exist_ok=True
    )


    # --------------------------------------------------------
    # Completed seed?
    # --------------------------------------------------------

    if summary_path(
        seed_output
    ).exists():

        print(
            f"\nSeed {seed}: "
            f"summary.json already exists."
        )

        print(
            "Skipping completed seed."
        )

        with open(
            summary_path(seed_output),
            "r"
        ) as f:

            return json.load(f)


    print("\n")
    print("=" * 70)
    print(
        f"STARTING SEED: {seed}"
    )
    print("=" * 70)


    # --------------------------------------------------------
    # Clean GPU before creating model
    # --------------------------------------------------------

    cleanup_gpu()

    print_gpu_memory(
        "Before model: "
    )


    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    (
        train_dataset,
        val_dataset,
        test_dataset
    ) = create_datasets()


    (
        train_loader,
        val_loader,
        test_loader
    ) = create_dataloaders(

        train_dataset,
        val_dataset,
        test_dataset
    )


    # --------------------------------------------------------
    # Seed
    # --------------------------------------------------------

    seed_everything(seed)


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = build_model(
        num_classes=num_classes
    ).to(DEVICE)


    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    if USE_CLASS_WEIGHTS:

        criterion = (
            nn.CrossEntropyLoss(
                weight=class_weights_tensor
            )
        )

    else:

        criterion = (
            nn.CrossEntropyLoss()
        )


    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY
    )


    # --------------------------------------------------------
    # Scheduler
    # --------------------------------------------------------

    scheduler = (
        optim.lr_scheduler.ReduceLROnPlateau(

            optimizer,

            mode="max",

            factor=LR_FACTOR,

            patience=LR_PATIENCE,

            min_lr=MIN_LR
        )
    )


    # --------------------------------------------------------
    # AMP scaler
    # --------------------------------------------------------

    scaler = GradScaler(

        enabled=(
            USE_AMP
            and DEVICE.type == "cuda"
        )
    )


    # --------------------------------------------------------
    # Resume
    # --------------------------------------------------------

    ckpt = load_checkpoint(
        seed_output
    )


    if ckpt is not None:

        print(
            f"\nSeed {seed}: "
            f"checkpoint found at "
            f"epoch {ckpt['epoch']}."
        )

        print(
            "Resuming..."
        )


        restore_checkpoint(

            ckpt,

            model,

            optimizer,

            scheduler,

            scaler
        )


        history = ckpt[
            "history"
        ]


        best_val_macro_f1 = ckpt[
            "best_val_macro_f1"
        ]


        best_epoch = ckpt[
            "best_epoch"
        ]


        best_state = ckpt[
            "best_state"
        ]


        patience_counter = ckpt[
            "patience_counter"
        ]


        start_epoch = (
            ckpt["epoch"] + 1
        )


        total_training_time_so_far = sum(

            h["epoch_time_sec"]

            for h in history
        )


        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        if start_epoch > EPOCHS:

            print(
                f"Seed {seed} already reached "
                f"the maximum epoch."
            )

            start_epoch = EPOCHS + 1


    else:

        print(
            f"\nSeed {seed}: "
            f"no checkpoint found."
        )

        print(
            "Starting fresh."
        )


        history = []

        best_val_macro_f1 = -np.inf

        best_epoch = 0

        best_state = None

        patience_counter = 0

        start_epoch = 1

        total_training_time_so_far = 0.0


    # --------------------------------------------------------
    # If already finished but summary missing
    # --------------------------------------------------------

    if start_epoch > EPOCHS:

        if best_state is None:

            raise RuntimeError(
                "Training appears complete but "
                "best_state is missing."
            )

    # --------------------------------------------------------
    # Training timer
    # --------------------------------------------------------

    start_time = time.time()

    stopped_early = False


    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(
        start_epoch,
        EPOCHS + 1
    ):

        epoch_start = time.time()

        model.train()


        running_loss = 0.0

        train_targets_epoch = []

        train_predictions_epoch = []


        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # Training batches
        # ----------------------------------------------------

        for step, (
            images,
            labels
        ) in enumerate(
            train_loader
        ):

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )


            is_accumulation_step = (

                (
                    step + 1
                )
                %
                GRAD_ACCUM_STEPS
                == 0

            ) or (

                (
                    step + 1
                )
                ==
                len(train_loader)
            )


            # ------------------------------------------------
            # Forward / backward
            # ------------------------------------------------

            try:

                with autocast(

                    device_type="cuda",

                    enabled=(
                        USE_AMP
                        and DEVICE.type
                        == "cuda"
                    )
                ):

                    outputs = model(
                        images
                    )

                    loss = (
                        criterion(
                            outputs,
                            labels
                        )
                        /
                        GRAD_ACCUM_STEPS
                    )


                scaler.scale(
                    loss
                ).backward()


            except torch.cuda.OutOfMemoryError:

                # --------------------------------------------
                # OOM recovery
                # --------------------------------------------

                print(
                    f"\n!! CUDA OOM "
                    f"at seed={seed}, "
                    f"epoch={epoch}, "
                    f"step={step}"
                )

                print(
                    "Clearing CUDA cache..."
                )


                # Remove current graph
                try:

                    del outputs

                except Exception:

                    pass


                try:

                    del loss

                except Exception:

                    pass


                del images
                del labels


                optimizer.zero_grad(
                    set_to_none=True
                )


                cleanup_gpu()


                print_gpu_memory(
                    "After OOM cleanup: "
                )


                print(
                    "\nThe current batch could "
                    "not fit in 4 GB VRAM."
                )

                print(
                    "Saving checkpoint at the "
                    "last COMPLETED epoch."
                )


                save_checkpoint(

                    seed_output,

                    seed,

                    epoch - 1,

                    model,

                    optimizer,

                    scheduler,

                    scaler,

                    history,

                    best_val_macro_f1,

                    best_epoch,

                    best_state,

                    patience_counter
                )


                print(
                    "\nCheckpoint saved."
                )

                print(
                    "Lower REAL_BATCH_SIZE to "
                    "8 and keep "
                    "GRAD_ACCUM_STEPS=4 if "
                    "this happens again."
                )

                raise


            # ------------------------------------------------
            # Optimizer step
            # ------------------------------------------------

            if is_accumulation_step:

                scaler.step(
                    optimizer
                )

                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )


            # ------------------------------------------------
            # Metrics
            # ------------------------------------------------

            running_loss += (
                loss.item()
                * images.size(0)
                * GRAD_ACCUM_STEPS
            )


            with torch.no_grad():

                predictions = (
                    torch.argmax(
                        outputs,
                        dim=1
                    )
                )


                train_targets_epoch.extend(

                    labels
                    .detach()
                    .cpu()
                    .numpy()
                )


                train_predictions_epoch.extend(

                    predictions
                    .detach()
                    .cpu()
                    .numpy()
                )


            # ------------------------------------------------
            # Release batch tensors
            # ------------------------------------------------

            del images
            del labels
            del outputs
            del predictions
            del loss


        # ----------------------------------------------------
        # Train metrics
        # ----------------------------------------------------

        train_loss = (
            running_loss
            /
            len(train_loader.dataset)
        )


        train_metrics = calculate_metrics(

            train_targets_epoch,

            train_predictions_epoch
        )


        # ----------------------------------------------------
        # Clean GPU before validation
        # ----------------------------------------------------

        del train_targets_epoch
        del train_predictions_epoch

        cleanup_gpu()


        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        val_metrics, _, _ = evaluate(

            model,

            val_loader,

            criterion
        )


        # ----------------------------------------------------
        # Clean GPU after validation
        # ----------------------------------------------------

        cleanup_gpu()


        # ----------------------------------------------------
        # Scheduler
        # ----------------------------------------------------

        scheduler.step(
            val_metrics[
                "macro_f1"
            ]
        )


        current_lr = (
            optimizer.param_groups[0]
            ["lr"]
        )


        # ----------------------------------------------------
        # Epoch time
        # ----------------------------------------------------

        epoch_time = (
            time.time()
            - epoch_start
        )


        row = {

            "seed":
                seed,

            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "train_accuracy":
                train_metrics[
                    "accuracy"
                ],

            "train_macro_f1":
                train_metrics[
                    "macro_f1"
                ],

            "train_weighted_f1":
                train_metrics[
                    "weighted_f1"
                ],

            "train_balanced_accuracy":
                train_metrics[
                    "balanced_accuracy"
                ],

            "val_loss":
                val_metrics[
                    "loss"
                ],

            "val_accuracy":
                val_metrics[
                    "accuracy"
                ],

            "val_macro_f1":
                val_metrics[
                    "macro_f1"
                ],

            "val_weighted_f1":
                val_metrics[
                    "weighted_f1"
                ],

            "val_balanced_accuracy":
                val_metrics[
                    "balanced_accuracy"
                ],

            "learning_rate":
                current_lr,

            "epoch_time_sec":
                epoch_time
        }


        history.append(
            row
        )


        # ----------------------------------------------------
        # Print
        # ----------------------------------------------------

        print(

            f"Seed {seed} | "
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss "
            f"{train_loss:.4f} | "
            f"Train F1 "
            f"{train_metrics['macro_f1']:.4f} | "
            f"Val Loss "
            f"{val_metrics['loss']:.4f} | "
            f"Val F1 "
            f"{val_metrics['macro_f1']:.4f} | "
            f"LR "
            f"{current_lr:.2e} | "
            f"{epoch_time:.1f}s"
        )


        # ----------------------------------------------------
        # Best model
        # ----------------------------------------------------

        if (

            val_metrics[
                "macro_f1"
            ]

            >

            best_val_macro_f1

        ):

            best_val_macro_f1 = (
                val_metrics[
                    "macro_f1"
                ]
            )

            best_epoch = epoch


            # CPU copy reduces GPU memory pressure
            best_state = {

                k:
                    v.detach()
                    .cpu()
                    .clone()

                if isinstance(
                    v,
                    torch.Tensor
                )

                else copy.deepcopy(v)

                for k, v
                in model.state_dict()
                .items()
            }


            torch.save(

                {

                    "seed":
                        seed,

                    "epoch":
                        epoch,

                    "model_state_dict":
                        best_state,

                    "class_names":
                        class_names,

                    "best_val_macro_f1":
                        best_val_macro_f1
                },

                seed_output
                / "best_model.pth"
            )


            patience_counter = 0


            print(

                f"  >>> BEST MODEL SAVED "
                f"(Val Macro-F1 = "
                f"{best_val_macro_f1:.4f})"
            )


        else:

            patience_counter += 1


        # ----------------------------------------------------
        # Checkpoint EVERY epoch
        # ----------------------------------------------------

        if (

            epoch
            %
            CHECKPOINT_EVERY_N_EPOCHS
            == 0

        ):

            save_checkpoint(

                seed_output,

                seed,

                epoch,

                model,

                optimizer,

                scheduler,

                scaler,

                history,

                best_val_macro_f1,

                best_epoch,

                best_state,

                patience_counter
            )


            print(
                "  >>> CHECKPOINT SAVED"
            )


        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if (

            patience_counter
            >=
            EARLY_STOPPING_PATIENCE

        ):

            print(

                f"  >>> Early stopping "
                f"at epoch {epoch}"
            )

            stopped_early = True


            save_checkpoint(

                seed_output,

                seed,

                epoch,

                model,

                optimizer,

                scheduler,

                scaler,

                history,

                best_val_macro_f1,

                best_epoch,

                best_state,

                patience_counter
            )


            break


        # ----------------------------------------------------
        # Memory report every epoch
        # ----------------------------------------------------

        print_gpu_memory(
            "  "
        )


    # ========================================================
    # TRAINING COMPLETE
    # ========================================================

    total_training_time = (

        total_training_time_so_far

        +

        (
            time.time()
            -
            start_time
        )
    )


    # ========================================================
    # RESTORE BEST MODEL
    # ========================================================

    if best_state is None:

        raise RuntimeError(
            f"No best model found "
            f"for seed {seed}."
        )


    # best_state is stored on CPU
    model.load_state_dict(
        best_state
    )


    model.to(DEVICE)


    cleanup_gpu()


    # ========================================================
    # TEST
    # ========================================================

    print(
        f"\nEvaluating BEST checkpoint "
        f"on TEST set for seed {seed}..."
    )


    (
        test_metrics,
        y_true,
        y_pred
    ) = evaluate(

        model,

        test_loader,

        criterion
    )


    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    report_dict = classification_report(

        y_true,

        y_pred,

        target_names=class_names,

        output_dict=True,

        zero_division=0
    )


    report_df = pd.DataFrame(
        report_dict
    ).transpose()


    report_df.to_csv(

        seed_output
        / "classification_report.csv"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(

        y_true,

        y_pred,

        labels=np.arange(
            num_classes
        )
    )


    cm_df = pd.DataFrame(

        cm,

        index=class_names,

        columns=class_names
    )


    cm_df.to_csv(

        seed_output
        / "confusion_matrix.csv"
    )


    # ========================================================
    # PLOTS
    # ========================================================

    if SAVE_PLOTS:

        # ----------------------------------------------------
        # Confusion matrix
        # ----------------------------------------------------

        plt.figure(
            figsize=(8, 7)
        )

        plt.imshow(
            cm,
            interpolation="nearest"
        )

        plt.title(
            f"Confusion Matrix — Seed {seed}"
        )

        plt.colorbar()

        tick_marks = np.arange(
            len(class_names)
        )

        plt.xticks(
            tick_marks,
            class_names,
            rotation=45,
            ha="right"
        )

        plt.yticks(
            tick_marks,
            class_names
        )

        threshold = (
            cm.max() / 2
            if cm.size > 0
            else 0
        )


        for i in range(
            cm.shape[0]
        ):

            for j in range(
                cm.shape[1]
            ):

                plt.text(

                    j,

                    i,

                    str(
                        cm[i, j]
                    ),

                    horizontalalignment=
                    "center",

                    verticalalignment=
                    "center",

                    color=(
                        "white"
                        if cm[i, j]
                        > threshold
                        else "black"
                    )
                )


        plt.ylabel(
            "True Label"
        )

        plt.xlabel(
            "Predicted Label"
        )

        plt.tight_layout()

        plt.savefig(

            seed_output
            / "confusion_matrix.png",

            dpi=300,

            bbox_inches="tight"
        )

        plt.close()


        # ----------------------------------------------------
        # Loss curve
        # ----------------------------------------------------

        history_df = pd.DataFrame(
            history
        )


        plt.figure(
            figsize=(8, 5)
        )

        plt.plot(

            history_df[
                "epoch"
            ],

            history_df[
                "train_loss"
            ],

            label="Train Loss"
        )

        plt.plot(

            history_df[
                "epoch"
            ],

            history_df[
                "val_loss"
            ],

            label="Validation Loss"
        )

        plt.xlabel(
            "Epoch"
        )

        plt.ylabel(
            "Loss"
        )

        plt.title(
            f"Loss Curve — Seed {seed}"
        )

        plt.legend()

        plt.tight_layout()

        plt.savefig(

            seed_output
            / "loss_curve.png",

            dpi=300,

            bbox_inches="tight"
        )

        plt.close()


        # ----------------------------------------------------
        # Macro F1 curve
        # ----------------------------------------------------

        plt.figure(
            figsize=(8, 5)
        )

        plt.plot(

            history_df[
                "epoch"
            ],

            history_df[
                "train_macro_f1"
            ],

            label="Train Macro-F1"
        )

        plt.plot(

            history_df[
                "epoch"
            ],

            history_df[
                "val_macro_f1"
            ],

            label="Validation Macro-F1"
        )

        plt.xlabel(
            "Epoch"
        )

        plt.ylabel(
            "Macro-F1"
        )

        plt.title(
            f"Macro-F1 Curve — Seed {seed}"
        )

        plt.legend()

        plt.tight_layout()

        plt.savefig(

            seed_output
            / "macro_f1_curve.png",

            dpi=300,

            bbox_inches="tight"
        )

        plt.close()


    else:

        history_df = pd.DataFrame(
            history
        )


    # ========================================================
    # SAVE TRAINING HISTORY
    # ========================================================

    history_df.to_csv(

        seed_output
        / "training_history.csv",

        index=False
    )


    # ========================================================
    # RESULT
    # ========================================================

    result = {

        "seed":
            seed,

        "best_epoch":
            best_epoch,

        "best_val_macro_f1":
            best_val_macro_f1,

        "test_accuracy":
            test_metrics[
                "accuracy"
            ],

        "test_macro_f1":
            test_metrics[
                "macro_f1"
            ],

        "test_weighted_f1":
            test_metrics[
                "weighted_f1"
            ],

        "test_balanced_accuracy":
            test_metrics[
                "balanced_accuracy"
            ],

        "training_time_sec":
            total_training_time,

        "num_epochs_run":
            len(history),

        "early_stopped":
            stopped_early
    }


    # ========================================================
    # SAVE SUMMARY
    # ========================================================

    with open(

        summary_path(
            seed_output
        ),

        "w"

    ) as f:

        json.dump(
            result,
            f,
            indent=4
        )


    # ========================================================
    # PRINT SEED RESULT
    # ========================================================

    print("\n")
    print("-" * 70)

    print(
        f"SEED {seed} TEST RESULTS"
    )

    print("-" * 70)

    print(
        f"Accuracy          : "
        f"{test_metrics['accuracy']:.4f}"
    )

    print(
        f"Macro-F1          : "
        f"{test_metrics['macro_f1']:.4f}"
    )

    print(
        f"Weighted-F1       : "
        f"{test_metrics['weighted_f1']:.4f}"
    )

    print(
        f"Balanced Accuracy : "
        f"{test_metrics['balanced_accuracy']:.4f}"
    )

    print(
        f"Best Epoch        : "
        f"{best_epoch}"
    )

    print(
        f"Training Time     : "
        f"{total_training_time / 60:.2f} min"
    )


    # ========================================================
    # FINAL GPU CLEANUP
    # ========================================================

    del model
    del optimizer
    del scheduler
    del scaler
    del best_state

    cleanup_gpu()


    return result


# ============================================================
# 20. RUN ALL FIVE SEEDS
# ============================================================

all_results = []

experiment_start = time.time()


for seed in SEEDS:

    try:

        result = train_one_seed(
            seed
        )

        all_results.append(
            result
        )

    except KeyboardInterrupt:

        print(
            "\nTraining manually interrupted."
        )

        print(
            "Existing checkpoint is preserved."
        )

        raise

    except torch.cuda.OutOfMemoryError:

        print(
            "\nCUDA OOM stopped the experiment."
        )

        print(
            "Your checkpoint has been saved."
        )

        print(
            "Restart the kernel and reduce:"
        )

        print(
            "REAL_BATCH_SIZE = 8"
        )

        print(
            "GRAD_ACCUM_STEPS = 4"
        )

        raise


# ============================================================
# 21. RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    all_results
)


results_df.to_csv(

    OUTPUT_ROOT
    / "5_seed_results.csv",

    index=False
)


# ============================================================
# 22. MEAN ± SD
# ============================================================

metric_columns = [

    "test_accuracy",

    "test_macro_f1",

    "test_weighted_f1",

    "test_balanced_accuracy",

    "best_val_macro_f1"
]


summary_rows = []


for metric in metric_columns:

    values = (
        results_df[metric]
        .astype(float)
    )


    summary_rows.append({

        "metric":
            metric,

        "mean":
            values.mean(),

        "std":
            values.std(
                ddof=1
            ),

        "min":
            values.min(),

        "max":
            values.max()
    })


mean_sd_df = pd.DataFrame(
    summary_rows
)


mean_sd_df.to_csv(

    OUTPUT_ROOT
    / "5_seed_mean_sd.csv",

    index=False
)


# ============================================================
# 23. FINAL EXPERIMENT SUMMARY
# ============================================================

total_experiment_time = (
    time.time()
    -
    experiment_start
)


final_summary = {

    "experiment":
        "OralVision-M4",

    "architecture":
        "EfficientNet-B0 + CoordAtt",

    "dataset":
        str(DATASET_ROOT),

    "image_size":
        IMAGE_SIZE,

    "real_batch_size":
        REAL_BATCH_SIZE,

    "grad_accum_steps":
        GRAD_ACCUM_STEPS,

    "effective_batch_size":
        EFFECTIVE_BATCH_SIZE,

    "amp_enabled":
        USE_AMP,

    "epochs_max":
        EPOCHS,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "dropout":
        DROPOUT,

    "coordatt_reduction":
        COORDATT_REDUCTION,

    "class_weights":
        USE_CLASS_WEIGHTS,

    "weighted_sampler":
        USE_WEIGHTED_SAMPLER,

    "seeds":
        SEEDS,

    "device":
        str(DEVICE),

    "num_classes":
        num_classes,

    "class_names":
        class_names,

    "train_distribution":
        train_distribution,

    "validation_distribution":
        val_distribution,

    "test_distribution":
        test_distribution,

    "total_experiment_time_sec":
        total_experiment_time
}


with open(

    OUTPUT_ROOT
    / "experiment_summary.json",

    "w"

) as f:

    json.dump(
        final_summary,
        f,
        indent=4
    )


# ============================================================
# 24. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)

print(
    "ORALVISION-M4 — "
    "FINAL 5-SEED RESULTS"
)

print("=" * 70)


print(

    results_df[[
        "seed",
        "best_epoch",
        "test_accuracy",
        "test_macro_f1",
        "test_weighted_f1",
        "test_balanced_accuracy"
    ]].to_string(
        index=False
    )
)


print("\n")
print("-" * 70)

print(
    "MEAN ± SD"
)

print("-" * 70)


for _, row in mean_sd_df.iterrows():

    print(

        f"{row['metric']:25s}: "
        f"{row['mean']:.4f} ± "
        f"{row['std']:.4f}"
    )


print("\n")
print("=" * 70)

print(
    "EXPERIMENT COMPLETE"
)

print("=" * 70)


print(
    f"Results saved to:\n"
    f"{OUTPUT_ROOT}"
)

print(
    "\nDone."
)

ORALVISION-M4 — RTX 2050 HARDENED VERSION
PyTorch version : 2.5.1+cu121
Torchvision     : 0.20.1+cu121
Device          : cuda
GPU             : NVIDIA GeForce RTX 2050
GPU memory      : 4.29 GB
GPU free memory : 3.37 GB / 4.29 GB
Dataset root    : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN
Output root     : C:\Users\R Y Z E N\Desktop\oralvision_M4_5seed_results
Real batch size : 16
Accumulation    : x2
Effective batch : 32
AMP enabled     : True

Dataset split directories:
TRAIN: C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\train
VAL  : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\val
TEST : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\test

Classes:
0: Calculus
1: Caries
2: Gingivitis
3: Hypodontia
4: Mouth_Ulcer
5: Tooth_Discoloration

Number of classes: 6

TRAIN distribution:
{'Calculus': 942, 'Caries': 5403, 'Gingivitis': 2595, 'Hypodontia': 2553, 'Mouth_Ulcer': 5844, 'Tooth_Discoloration': 363}

VALIDATION distribution:
{'Calcu